In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


I0000 00:00:1788814969.046878   19659 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788814970.369914   19659 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788814974.066767   19659 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
data_path = "../data/processed/tienda_tecnologica_limpio.csv"
df = pd.read_csv(data_path)

In [ ]:
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print("Columnas:", df.columns.tolist())

In [ ]:
X_raw = df["opinion_usuario"].fillna("").astype(str)
y = df["sentimiento"].values

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words="spanish")
X_vectorized = vectorizer.fit_transform(X_raw).toarray()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nPérdida (Loss): {loss:.4f}")
print(f"Precisión (Accuracy): {accuracy:.4f}")

In [ ]:
os.makedirs("../models", exist_ok=True)

In [ ]:
model.save("../models/neural_network.h5")

In [ ]:
with open("../models/tfidf_vectorizer_dl.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
print("¡Archivos guardados en 'models/neural_network.h5' y 'models/tfidf_vectorizer_dl.pkl'!")